In [1]:
import transformers
import accelerate
import torch
import sys

print(sys.executable)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("Torch:", torch.__version__)

c:\Users\Ahana Arora\OneDrive\Pictures\Documents\AI-Bug-Severity-Predictor\venv\Scripts\python.exe
Transformers: 4.52.4
Accelerate: 1.14.0
Torch: 2.11.0+cu128


In [2]:
import sys
print(sys.executable)

c:\Users\Ahana Arora\OneDrive\Pictures\Documents\AI-Bug-Severity-Predictor\venv\Scripts\python.exe


In [3]:
import torch
print(f"Is GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Is GPU available: True
GPU Name: NVIDIA GeForce RTX 3050 Laptop GPU


In [ ]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)
print(torch.cuda.get_device_name(0))

cuda
NVIDIA GeForce RTX 3050 Laptop GPU


In [31]:
df = pd.read_csv("../../data/processed/final_dataset.csv")

In [32]:
print(df.columns.tolist())

['summary', 'description', 'priority.name', 'status.name', 'resolution.name', 'issuetype.name']


In [33]:
df.head()

,summary,description,priority.name,status.name,resolution.name,issuetype.name
0,Update config browser to work with the new syntax,The config browser used Velocity calling the t...,Minor,Closed,Fixed,Improvement
1,XALAN_C 1.9 or current do not build on Fedora ...,Two types of errors:\n1- runConfigure and conf...,Blocker,Resolved,Fixed,Bug
2,"Problem with ADD new post, and DELETE post.","When trying to add new post, I was getting nex...",Critical,Closed,Cannot Reproduce,Bug
3,LogHandler can only work in GlobalConfiguratio...,org.apache.axis.handlers.LogHandler in request...,Major,Open,Unknown,Bug
4,Decoding of service is broken in org.apache.ax...,The following code assumes a lot of things:\n\...,Major,Open,Unknown,Bug


In [34]:
df["text"] = (
    df["summary"].fillna("") + " " +
    df["description"].fillna("")
)

In [35]:
df["target"] = df["priority.name"]

In [36]:
print(df["target"].value_counts())

target
Major             746289
Minor             216620
Critical           49514
Blocker            36476
Unknown            31859
Trivial            30399
Normal             13556
Low                 6829
P2                  6781
P3                  6372
Not a Priority      2365
P1                   838
Urgent               676
P4                   298
P0                   274
High                 177
Name: count, dtype: int64


In [37]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

df["target"] = label_encoder.fit_transform(df["target"])

In [38]:
print(df[["text", "target"]].head())

                                                text  target
0  Update config browser to work with the new syn...       5
1  XALAN_C 1.9 or current do not build on Fedora ...       0
2  Problem with ADD new post, and DELETE post. Wh...       1
3  LogHandler can only work in GlobalConfiguratio...       4
4  Decoding of service is broken in org.apache.ax...       4


In [39]:
df = df.sample(
    n=100000,
    random_state=42
).reset_index(drop=True)

In [40]:
print(df.shape)

(100000, 8)


In [41]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df[["text", "target"]],
    test_size=0.2,
    random_state=42,
    stratify=df["target"]
)

In [42]:
max_length = 128

In [43]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [44]:
print(label_encoder.classes_)
print(len(label_encoder.classes_))

['Blocker' 'Critical' 'High' 'Low' 'Major' 'Minor' 'Normal'
 'Not a Priority' 'P0' 'P1' 'P2' 'P3' 'P4' 'Trivial' 'Unknown' 'Urgent']
16


In [45]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=5
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [46]:
df = df[["text", "target"]]

In [47]:
df = df.dropna(subset=["text"])
df["text"] = df["text"].astype(str)

In [48]:
print(df["target"].value_counts())

target
4     64790
5     18899
1      4269
0      3200
14     2811
13     2736
6      1146
11      602
10      569
3       558
7       219
9        81
15       60
12       25
8        21
2        14
Name: count, dtype: int64


In [49]:
df = df.sample(
    n=100000,
    random_state=42
)

In [50]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [51]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [52]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding=True,
        max_length=128
    )

In [53]:
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/80000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [54]:
train_dataset = train_dataset.remove_columns(["text", "__index_level_0__"])
test_dataset = test_dataset.remove_columns(["text", "__index_level_0__"])

In [56]:
train_dataset = train_dataset.rename_column("target", "labels")
test_dataset = test_dataset.rename_column("target", "labels")

In [57]:
train_dataset.set_format("torch")
test_dataset.set_format("torch")

In [58]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=16
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [59]:
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [60]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [61]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted"
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [62]:
import transformers
import accelerate

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)

Transformers: 4.52.4
Accelerate: 1.14.0


In [63]:
print(len(df))

100000


In [64]:
import transformers
print(transformers.__version__)

4.52.4


In [65]:
print(df["target"].unique())
print(df["target"].min())
print(df["target"].max())
print(df["target"].dtype)

[ 4  5  0  1 13 14 10  6  3  7 11  9 15  8 12  2]
0
15
int64


In [66]:
print(df["target"].nunique())

16


In [67]:
import transformers
import accelerate
import torch

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("Torch:", torch.__version__)

Transformers: 4.52.4
Accelerate: 1.14.0
Torch: 2.11.0+cu128


In [68]:
from transformers import TrainingArguments

print("TrainingArguments imported successfully")

TrainingArguments imported successfully


In [69]:
import transformers
print(transformers.__version__)

4.52.4


In [70]:
print(train_dataset[0])

{'labels': tensor(4), 'input_ids': tensor([  101,  1031,  1039,  1009,  1009,  1033, 16215, 16338,  1035,  4958,
         1014,  1012,  2260,  1012,  1014, 11896,  2000,  3857,  2043,  2478,
         8612,  1035, 12992,  1035, 21431,  2098,  1027,  2006,  1045,  3603,
         2023,  2043,  2551,  2006,  8612,  1011,  4724, 23777,  1006,  1057,
         8569,  3372,  2226,  2403,  1012,  5840,  3857,  1010,  2073,  1996,
         2291, 12992,  2003,  4895,  6342,  9397, 15613,  1007,  1012,   102,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0, 

In [71]:
print("Number of classes:", df["target"].nunique())
print("Unique labels:", sorted(df["target"].unique()))

Number of classes: 16
Unique labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15)]


In [72]:
labels = train_dataset["labels"]

print("Minimum label:", min(labels))
print("Maximum label:", max(labels))
print("Unique labels:", sorted(set(labels)))

Minimum label: tensor(0)
Maximum label: tensor(15)
Unique labels: [tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0), tensor(0)

In [73]:
labels = train_dataset["labels"]

invalid = [x for x in labels if x < 0 or x >= 16]

print("Invalid labels:", invalid[:20])
print("Count:", len(invalid))

Invalid labels: []
Count: 0


In [74]:
import torch

print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory / 1024**3)

NVIDIA GeForce RTX 3050 Laptop GPU
3.99951171875


In [75]:
import transformers
import accelerate
import torch

print(transformers.__version__)
print(accelerate.__version__)
print(torch.__version__)

4.52.4
1.14.0
2.11.0+cu128


In [76]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [77]:
label_encoder.fit_transform(df["target"])

array([4, 4, 4, ..., 4, 4, 4], shape=(100000,))

In [79]:
print(train_dataset.features)

{'labels': Value('int64'), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}


In [80]:
print(next(model.parameters()).device)

cuda:0


In [82]:
train_dataset.save_to_disk(r"C:\AI-Bug-Severity-Predictor\train_dataset")
test_dataset.save_to_disk(r"C:\AI-Bug-Severity-Predictor\test_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/80000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/20000 [00:00<?, ? examples/s]

In [83]:
from datasets import load_from_disk

train_dataset = load_from_disk("train_dataset")
test_dataset = load_from_disk("test_dataset")

In [84]:
from datasets import load_from_disk

train_dataset = load_from_disk("train_dataset")
test_dataset = load_from_disk("test_dataset")

In [85]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 3050 Laptop GPU


In [86]:
from transformers import BertForSequenceClassification

device = torch.device("cuda")

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=16
)

model = model.to(device)

print(next(model.parameters()).device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


cuda:0


In [87]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [88]:
sample = train_dataset[0]

print(type(sample["input_ids"]))
print(type(sample["attention_mask"]))
print(type(sample["token_type_ids"]))
print(type(sample["labels"]))

<class 'torch.Tensor'>
<class 'torch.Tensor'>
<class 'torch.Tensor'>
<class 'torch.Tensor'>


In [89]:
print(sample)

{'labels': tensor(4), 'input_ids': tensor([  101,  1031,  1039,  1009,  1009,  1033, 16215, 16338,  1035,  4958,
         1014,  1012,  2260,  1012,  1014, 11896,  2000,  3857,  2043,  2478,
         8612,  1035, 12992,  1035, 21431,  2098,  1027,  2006,  1045,  3603,
         2023,  2043,  2551,  2006,  8612,  1011,  4724, 23777,  1006,  1057,
         8569,  3372,  2226,  2403,  1012,  5840,  3857,  1010,  2073,  1996,
         2291, 12992,  2003,  4895,  6342,  9397, 15613,  1007,  1012,   102,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0, 

In [90]:
batch = {
    "input_ids": sample["input_ids"].unsqueeze(0).to(device),
    "attention_mask": sample["attention_mask"].unsqueeze(0).to(device),
    "token_type_ids": sample["token_type_ids"].unsqueeze(0).to(device),
    "labels": sample["labels"].unsqueeze(0).to(device),
}

In [91]:
outputs = model(**batch)

print(outputs.loss)

tensor(2.6446, device='cuda:0', grad_fn=<NllLossBackward0>)


In [92]:
print(model.config.num_labels)

16


In [93]:
training_args = TrainingArguments(
    output_dir="../../models/bert_output",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=4,

    num_train_epochs=3,

    weight_decay=0.01,

    logging_steps=100,

    load_best_model_at_end=True,

    fp16=False,

    report_to="none"
)

In [94]:
trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    tokenizer=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

C:\Users\Ahana Arora\AppData\Local\Temp\ipykernel_6084\260658729.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [95]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.007700,0.972711,0.674650,0.589150,0.674650,0.579492
2,0.872700,0.941374,0.670200,0.612778,0.670200,0.618586
3,0.671900,1.022570,0.663150,0.612474,0.663150,0.622396


c:\Users\Ahana Arora\OneDrive\Pictures\Documents\AI-Bug-Severity-Predictor\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Ahana Arora\OneDrive\Pictures\Documents\AI-Bug-Severity-Predictor\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Ahana Arora\OneDrive\Pictures\Documents\AI-Bug-Severity-Predictor\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted sampl

TrainOutput(global_step=30000, training_loss=0.8867394630432129, metrics={'train_runtime': 15165.4174, 'train_samples_per_second': 15.825, 'train_steps_per_second': 1.978, 'total_flos': 1.578864771072e+16, 'train_loss': 0.8867394630432129, 'epoch': 3.0})

In [96]:
results = trainer.evaluate()

print(results)

{'eval_loss': 0.9413742423057556, 'eval_accuracy': 0.6702, 'eval_precision': 0.6127783915044112, 'eval_recall': 0.6702, 'eval_f1': 0.6185863885531733, 'eval_runtime': 408.7189, 'eval_samples_per_second': 48.933, 'eval_steps_per_second': 24.467, 'epoch': 3.0}


c:\Users\Ahana Arora\OneDrive\Pictures\Documents\AI-Bug-Severity-Predictor\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [97]:
trainer.save_model("../../models/bert_model")
tokenizer.save_pretrained("../../models/bert_model")

('../../models/bert_model\\tokenizer_config.json',
 '../../models/bert_model\\special_tokens_map.json',
 '../../models/bert_model\\vocab.txt',
 '../../models/bert_model\\added_tokens.json')

In [98]:
model.save_pretrained("../../models/bert_model")
tokenizer.save_pretrained("../../models/bert_model")

('../../models/bert_model\\tokenizer_config.json',
 '../../models/bert_model\\special_tokens_map.json',
 '../../models/bert_model\\vocab.txt',
 '../../models/bert_model\\added_tokens.json')